# Quick Navigation Table

Use the links below to quickly navigate to the corresponding cells in this notebook:

| Cell Name | Description | Link |
| :--- | :--- | :--- |
| **Cell 1** | Drive Mount & Data Unzipping | [Go to Cell 1](#cell_WNVMBumhWWtB) |
| **Cell 2** | Libraries & Dependencies Imports | [Go to Cell 2](#cell_ZEzBRbM3A2yj) |
| **Cell 3** | Data Loading & Shuffling | [Go to Cell 3](#cell_MRKrdqgdA2ym) |
| **Cell 4** | Feature Encoding (Gender) | [Go to Cell 4](#cell_TIjUfMuJA2yn) |
| **Cell 5** | Explore Data Columns | [Go to Cell 5](#cell_RwBzWS-IA2yo) |
| **Cell 6** | Feature/Target Splitting | [Go to Cell 6](#cell_fZ1Abm_yA2yp) |
| **Cell 7** | Train/Test Set Splitting | [Go to Cell 7](#cell_0unCh_ZAA2yq) |
| **Cell 8** | Model Creation & Evaluation | [Go to Cell 8](#cell_Q-6vXZSQA2yq) |
| **Cell 9** | Model Selection | [Go to Cell 9](#cell_GUCCz779A2ys) |
| **Cell 10** | Validation Predictions Export | [Go to Cell 10](#cell_Qfot1W1cEDCp) |

# Lab 3 Exercise 3
## Binary Classification

We are given a dataset that consists of biological features of a person and whether or not they are a smoker.

There are *22 descriptive features* and *1 target column* in the dataset as described in the Table below. You are tasked with creating a predictive model to predict the column "SMK_stat_type_cd", which specifies smoker/non-smoker status of the person. You are expected to use classification methods that was shown in the previous exercises.



|      Column      |                                 Description                                 |
|:----------------:|:---------------------------------------------------------------------------:|
| identified_gender              | male, female                                                                |
| age              | round up to 5 years                                                         |
| height           | round up to 5 cm[cm]                                                        |
| weight           | [kg]                                                                        |
| sight_left       | eyesight(left)                                                              |
| sight_right      | eyesight(right)                                                             |
| hear_left        | hearing left, 1(normal), 2(abnormal)                                        |
| hear_right       | hearing right, 1(normal), 2(abnormal)                                       |
| SBP              | Systolic blood pressure[mmHg]                                               |
| DBP              | Diastolic blood pressure[mmHg]                                              |
| BLDS             | BLDS or FSG(fasting blood glucose)[mg/dL]                                   |
| tot_chole        | total cholesterol[mg/dL]                                                    |
| HDL_chole        | HDL cholesterol[mg/dL]                                                      |
| LDL_chole        | LDL cholesterol[mg/dL]                                                      |
| triglyceride     | triglyceride[mg/dL]                                                         |
| hemoglobin       | hemoglobin[g/dL]                                                            |
| urine_protein    | protein in urine, 1(-), 2(+/-), 3(+1), 4(+2), 5(+3), 6(+4)                  |
| serum_creatinine | serum(blood) creatinine[mg/dL]                                              |
| SGOT_AST         | SGOT(Glutamate-oxaloacetate transaminase) AST(Aspartate transaminase)[IU/L] |
| SGOT_ALT         | ALT(Alanine transaminase)[IU/L]                                             |
| gamma_GTP        | y-glutamyl transpeptidase[IU/L]                                             |
| SMK_stat_type_cd **(Target)** | Smoking state, 0(never), 1(active smoker)                      |


### The stages of this problem can be decomposed as follows:
1. Data Preparation
* Ensure data is in correct format (numerical and not string)
* Normalize the data for better convergence (optional)
* Split the data into train/test subsets

2. Model Selection
* Instanciate models and fit on training data
* Evaluate model performance on testing data
* Select model with best performance

3. Submit model
* Your model will be evaluated on data that is kept separate from training/testing data
* The predictions from your model will be uploaded to the course server where it will be evaluated

In [1]:
# Cell 1
from google.colab import drive
drive.mount('/content/gdrive')
%cd /content/gdrive/MyDrive/4c16-labs/code/dist/lab-03/
!unzip -o data.zip
!echo 'data/*' > .gitignore

Mounted at /content/gdrive
/content/gdrive/MyDrive/4c16-labs/code/dist/lab-03
Archive:  data.zip
   creating: data/
  inflating: data/data.csv           
  inflating: data/validation.csv     


In [2]:
# Cell 2
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sklearn.preprocessing
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay

In [3]:
# Cell 3
# Data Preparation
# The data is supplied as ".csv" format
# There are 23 columns in csv file, with 22 columns being features and 1 column being the target

# The csv file can be read into as a dataframe using the pandas library
DataFrame = pd.read_csv('data/data.csv')

# Shuffle the data
DataFrame = DataFrame.sample(frac=1)

# Visualize the first 5 rows of the imported dataframe
display(DataFrame.head(6))

,identified_gender,age,height,weight,waistline,sight_left,sight_right,hear_left,hear_right,SBP,...,HDL_chole,LDL_chole,triglyceride,hemoglobin,urine_protein,serum_creatinine,SGOT_AST,SGOT_ALT,gamma_GTP,SMK_stat_type_cd
4131,Male,40,170,75,93.0,1.5,1.5,1.0,1.0,136.0,...,60.0,122.0,161.0,14.5,1.0,0.9,24.0,29.0,22.0,0.0
46694,Male,40,170,65,82.0,1.0,1.0,1.0,1.0,130.0,...,48.0,156.0,108.0,15.7,1.0,1.0,16.0,20.0,25.0,1.0
38247,Female,60,155,70,89.0,0.7,0.5,1.0,1.0,135.0,...,39.0,116.0,131.0,13.0,1.0,1.2,43.0,46.0,35.0,0.0
676,Female,75,155,65,95.0,0.1,0.6,1.0,1.0,130.0,...,53.0,80.0,169.0,13.6,1.0,0.8,31.0,17.0,17.0,0.0
19808,Female,55,150,50,68.0,0.9,1.2,1.0,1.0,126.0,...,65.0,185.0,108.0,13.8,1.0,0.6,25.0,27.0,17.0,0.0
22102,Male,35,175,90,93.0,1.5,1.5,1.0,1.0,130.0,...,40.0,116.0,480.0,16.1,1.0,0.8,29.0,29.0,114.0,1.0


In [4]:
# Cell 4
# The column "identified_gender" contains non-numeric data. This cannot be used as is and needs to be converted to numerical representations
# We can encode the identified gender to a numerical representation by letting "female" == 0, and "male" == 1.
# This can be done manually by using conditional statements on the dataframe, but luckily for us there is a simpler method
# We can use the built in LabelEncoder function of sklearn to do this

LabelEncoder = sklearn.preprocessing.LabelEncoder()
LabelEncoder.fit(DataFrame['identified_gender'])
DataFrame['identified_gender'] = LabelEncoder.transform(DataFrame['identified_gender'])

# Label encoder sorts the input column of data in alphabetical order, and then assigns a numerical value to each unique entry
# This results in 'female' being mapped to 0, 'male' being mapped to 1
display(DataFrame.head(6))

,identified_gender,age,height,weight,waistline,sight_left,sight_right,hear_left,hear_right,SBP,...,HDL_chole,LDL_chole,triglyceride,hemoglobin,urine_protein,serum_creatinine,SGOT_AST,SGOT_ALT,gamma_GTP,SMK_stat_type_cd
4131,1,40,170,75,93.0,1.5,1.5,1.0,1.0,136.0,...,60.0,122.0,161.0,14.5,1.0,0.9,24.0,29.0,22.0,0.0
46694,1,40,170,65,82.0,1.0,1.0,1.0,1.0,130.0,...,48.0,156.0,108.0,15.7,1.0,1.0,16.0,20.0,25.0,1.0
38247,0,60,155,70,89.0,0.7,0.5,1.0,1.0,135.0,...,39.0,116.0,131.0,13.0,1.0,1.2,43.0,46.0,35.0,0.0
676,0,75,155,65,95.0,0.1,0.6,1.0,1.0,130.0,...,53.0,80.0,169.0,13.6,1.0,0.8,31.0,17.0,17.0,0.0
19808,0,55,150,50,68.0,0.9,1.2,1.0,1.0,126.0,...,65.0,185.0,108.0,13.8,1.0,0.6,25.0,27.0,17.0,0.0
22102,1,35,175,90,93.0,1.5,1.5,1.0,1.0,130.0,...,40.0,116.0,480.0,16.1,1.0,0.8,29.0,29.0,114.0,1.0


In [5]:
# Cell 5
# We should now separate the input features from the target feature and store them as different variables
# We do this by slicing what columns of data we want from the dataframe
# Let's first see the columns available to us by printing DataFrame.columns
print(DataFrame.columns)

Index(['identified_gender', 'age', 'height', 'weight', 'waistline',
       'sight_left', 'sight_right', 'hear_left', 'hear_right', 'SBP', 'DBP',
       'BLDS', 'tot_chole', 'HDL_chole', 'LDL_chole', 'triglyceride',
       'hemoglobin', 'urine_protein', 'serum_creatinine', 'SGOT_AST',
       'SGOT_ALT', 'gamma_GTP', 'SMK_stat_type_cd'],
      dtype='object')


In [6]:
# Cell 6
# The column titled "SMK_stat_type_cd" is our target, and all remaining variables are our input features.
features_columns = DataFrame.columns[:-1]
target_column = DataFrame.columns[-1:]

DataFrame_X = DataFrame[features_columns]   # Selects only Input features
DataFrame_Y = DataFrame[target_column]      # Selects only Target variable

In [7]:
# Cell 7
# Split Data into Train/Test split
# Change the variable "test_frac" to reflect the percentage/fraction of test data in the resulting split
test_frac = 0.15

Train_X, Test_X, Train_Y, Test_Y = train_test_split(DataFrame_X, DataFrame_Y, test_size=test_frac)
print(f"Number of rows for Training:\t{Train_X.shape[0]}\nNumber of rows for Testing:\t{Test_X.shape[0]}")

Number of rows for Training:	42500
Number of rows for Testing:	7500


In [8]:
# Cell 8
# EDIT THIS CELL
# Construct a dictionary of prediction models to compare
# Uncomment the below dictionary and insert as many prediction models as you like.
# You may have used binary classification models in previous exercises
# You may also have to import these modules/libraries to be able to use them

from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

PredictionModels = {
    'Logistic Regression': make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000),
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced',
        random_state=0,
        n_jobs=-1,
    ),
    'Histogram Gradient Boosting': HistGradientBoostingClassifier(
        max_iter=200,
        class_weight='balanced',
        random_state=0,
    ),
}

models = list(PredictionModels.keys())
train_f1_scores = {}
test_f1_scores = {}

print("Fitting models, this may take a while")
for _model in models:
    model = PredictionModels[_model]
    model.fit(Train_X, Train_Y.values.ravel())

    train_predictions = model.predict(Train_X)
    test_predictions = model.predict(Test_X)
    train_f1_scores[_model] = f1_score(Train_Y.values.ravel(), train_predictions)
    test_f1_scores[_model] = f1_score(Test_Y.values.ravel(), test_predictions)

    print(
        f"{_model}: train F1 = {train_f1_scores[_model]:.4f}, "
        f"test F1 = {test_f1_scores[_model]:.4f}"
    )

Fitting models, this may take a while
Logistic Regression: train F1 = 0.7498, test F1 = 0.7531
Random Forest: train F1 = 1.0000, test F1 = 0.7452
Histogram Gradient Boosting: train F1 = 0.7786, test F1 = 0.7625


In [9]:
# Cell 9
# Select the model with best f1 score and write down the
# corresponding 'key' value from the 'PredictionModels' variable
# Eg if Logistic Regression is chosen then: chosen_model = 'Logistic Regression'
chosen_model = 'Logistic Regression'

In [10]:
# Cell 10
# DO NOT EDIT.
# Generate predictions using "chosen_model" and save to file

backend_features = pd.read_csv('data/validation.csv')
backend_features['identified_gender'] = LabelEncoder.transform(backend_features['identified_gender'])
backend_preds = PredictionModels[chosen_model].predict(backend_features)
np.savez_compressed('lab3_ex3_preds', lab3_model=backend_preds)

# Remember to push your changes to the git server for marking!